# Exercise 16.1: Solving the bistable equation in Python

We want to solve the bistable equation on a 1D cable of length $L = 100$:

$$
\frac{\partial V}{\partial t} = k \frac{\partial^2 V}{\partial x^2} + A V(1 - V)(V - \alpha)
$$

We will use the following parameter values:

- $k = 2.0$ (Diffusion constant)
- $A = 1.0$ (Reaction scaling)
- $\alpha = 0.1$ (Activation threshold)
- $\Delta x = 1$ (Spatial step)
- $\Delta t = 0.1$ ms (Time step)

We will apply an initial stimulus to the far left edge of the cable ($V = 0.3$ for the first 10% of the cable) to trigger the wave. The boundary conditions at the absolute ends of the cable ($x=0$ and $x=L$) will be sealed (no current can flow out the ends).


## Exercise 16.1a: Explicit scheme with loops

Complete the explicit update scheme inside the loop below.

You need to use the `v_prev` array (which holds the voltage from the previous time step $n$) to calculate the spatial diffusion and the reaction term, and then save the new values into the `v` array (which represents step $n+1$).

_Note: The boundary conditions at $j=0$ and $j=N$ have been provided for you to handle the sealed ends._


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters
k = 2.0
A = 1.0
alpha = 0.1
L = 100

dx = 1
dt = 0.1
N = int(L / dx)
n_steps = 1400

# Initialize arrays
v = np.zeros(N + 1)
left = int(N / 10)
v[:left] = 0.3  # Apply the initial stimulus to the left side!


def f(V):
    return A * V * (1 - V) * (V - alpha)


# We use v_prev to hold the known values from the previous time step
v_prev = np.copy(v)

# Simulation loop
for i in range(n_steps):
    # 1. Update the internal nodes using the explicit FDM scheme
    for j in range(1, N):
        # Your code here: calculate I_ion, diffusion, and the new v[j]
        I_ion = ...
        diff = ...
        v[j] = ...

    # 2. Update the boundary nodes (sealed ends)
    v[0] = (
        v_prev[0] + dt * (k / dx**2) * 2 * (v_prev[1] - v_prev[0]) + dt * f(v_prev[0])
    )
    v[N] = (
        v_prev[N]
        + dt * (k / dx**2) * 2 * (v_prev[N - 1] - v_prev[N])
        + dt * f(v_prev[N])
    )

    # 3. Save the newly calculated state for the next loop iteration
    v_prev = np.copy(v)

**Observation question:** After running the simulation, plot the final state of `v` and describe what you see. The wave has travelled from the left edge of the cable to the right. What is the approximate wave speed (in grid points per time step)? Does the shape of the wavefront change as it propagates?


In [ ]:
# Plot the final state
plt.figure(figsize=(10, 4))
plt.plot(v, color="C0", linewidth=2)
plt.xlabel("Position (x)")
plt.ylabel("Voltage (V)")
plt.title(f"Bistable equation — final state after {n_steps} time steps")
plt.ylim(-0.1, 1.2)
plt.show()

## Exercise 16.1b: Vectorization for speed

Standard `for` loops in Python are notoriously slow. In computational physiology, tracking thousands of grid points using Python loops can bring your simulation to a grinding halt.

Instead, we can use **NumPy vectorization**. By passing entire slices of arrays into our FDM formula at once, NumPy handles the loop internally in highly optimized C-code, speeding up the calculation by orders of magnitude!

Complete the code below to implement the exact same scheme, but without the internal spatial `for` loop. We have set up the slicing indices `I`, `Ip` (I plus 1), and `Im` (I minus 1) for you.


In [ ]:
# Reset initial conditions
v = np.zeros(N + 1)
v[:left] = 0.3

# Introduce the appropriate arrays for slicing the internal nodes
I = np.arange(1, N)  # Current nodes
Ip = I + 1  # Right neighbors
Im = I - 1  # Left neighbors

for i in range(n_steps):
    # 1. Calculate the reaction term for ALL nodes simultaneously
    I_ion = ...

    # We must explicitly save the old state before doing vectorized in-place updates!
    v_prev = np.copy(v)

    # 2. Add diffusion to the internal nodes using array slicing
    v[I] = v_prev[I] + dt * (k / dx**2) * (...)

    # 3. Update the boundary nodes
    v[0] = v_prev[0] + dt * (k / dx**2) * 2 * (v_prev[1] - v_prev[0])
    v[N] = v_prev[N] + dt * (k / dx**2) * 2 * (v_prev[N - 1] - v_prev[N])

    # 4. Add the reaction term to the entire array at once
    v = v + dt * I_ion

### Visualizing the time evolution

To properly observe the travelling wave, it is helpful to store the solution at regular intervals and then use an interactive slider to scrub through time.

Run the cell below to simulate the bistable equation with the vectorized scheme and explore the time evolution interactively.


In [ ]:
from ipywidgets import interact, IntSlider

# Simulate and store snapshots
save_every = 10
v = np.zeros(N + 1)
v[:left] = 0.3

I = np.arange(1, N)
Ip = I + 1
Im = I - 1

snapshots = [v.copy()]

for i in range(n_steps):
    I_ion = f(v)
    v_prev = np.copy(v)
    v[I] = v_prev[I] + dt * (k / dx**2) * (v_prev[Ip] - 2 * v_prev[I] + v_prev[Im])
    v[0] = v_prev[0] + dt * (k / dx**2) * 2 * (v_prev[1] - v_prev[0])
    v[N] = v_prev[N] + dt * (k / dx**2) * 2 * (v_prev[N - 1] - v_prev[N])
    v = v + dt * I_ion
    if (i + 1) % save_every == 0:
        snapshots.append(v.copy())

snapshots = np.array(snapshots)
x = np.linspace(0, L, N + 1)


# Interactive slider
def plot_bistable(frame=0):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(x, snapshots[frame], color="C0", linewidth=2)
    ax.set_xlim(0, L)
    ax.set_ylim(-0.1, 1.2)
    ax.set_xlabel("Position (x)")
    ax.set_ylabel("Voltage (V)")
    t_ms = frame * save_every * dt
    ax.set_title(
        f"Bistable equation — t = {t_ms:.1f} ms (frame {frame}/{len(snapshots) - 1})"
    )
    plt.show()


interact(
    plot_bistable,
    frame=IntSlider(
        min=0, max=len(snapshots) - 1, step=1, value=0, description="Time frame"
    ),
)

## Exercise 16.1c: Exploring the parameters (Widget)

The behaviour of the bistable equation depends critically on three parameters:

- **$\alpha$ (threshold):** Below this value, the reaction term pulls the voltage back to 0. Above it, the voltage is driven toward 1.
- **$k$ (diffusion coefficient):** Controls how quickly voltage diffuses spatially.
- **$A$ (reaction scaling):** Controls the strength of the cubic reaction term.

Use the widget below to explore how these parameters affect the travelling wave.

**Questions to investigate:**

1. What happens to the wave if you increase $\alpha$ from 0.1 toward 0.5? At what approximate value does the wave fail to propagate? Why?
2. How does the wave speed change if you increase $k$? What about $A$?
3. With $\alpha = 0.1$ and small $k$, is there a combination where the simulation becomes unstable? (Hint: recall the stability condition $r = k \Delta t / \Delta x^2 \leq 0.5$)


In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider


def run_bistable(alpha_w=0.1, k_w=2.0, A_w=1.0, frame=0):
    """Run the bistable equation and display a specific time frame."""
    dx_w = 1
    dt_w = 0.1
    L_w = 100
    N_w = int(L_w / dx_w)
    n_steps_w = 1400
    save_every_w = 10

    def f_w(V):
        return A_w * V * (1 - V) * (V - alpha_w)

    v = np.zeros(N_w + 1)
    v[: int(N_w / 10)] = 0.3

    I_arr = np.arange(1, N_w)
    Ip_arr = I_arr + 1
    Im_arr = I_arr - 1

    snaps = [v.copy()]
    for step in range(n_steps_w):
        I_ion_w = f_w(v)
        vp = np.copy(v)
        v[I_arr] = vp[I_arr] + dt_w * (k_w / dx_w**2) * (
            vp[Ip_arr] - 2 * vp[I_arr] + vp[Im_arr]
        )
        v[0] = vp[0] + dt_w * (k_w / dx_w**2) * 2 * (vp[1] - vp[0])
        v[N_w] = vp[N_w] + dt_w * (k_w / dx_w**2) * 2 * (vp[N_w - 1] - vp[N_w])
        v = v + dt_w * I_ion_w
        if (step + 1) % save_every_w == 0:
            snaps.append(v.copy())

    # Clamp frame index
    frame = min(frame, len(snaps) - 1)

    fig, ax = plt.subplots(figsize=(10, 4))
    x_arr = np.linspace(0, L_w, N_w + 1)
    ax.plot(x_arr, snaps[frame], color="C0", linewidth=2)
    ax.set_xlim(0, L_w)
    ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel("Position (x)")
    ax.set_ylabel("Voltage (V)")
    r_val = k_w * dt_w / dx_w**2
    ax.set_title(
        f"α={alpha_w:.2f}, k={k_w:.1f}, A={A_w:.1f} | r={r_val:.2f} | "
        f"t = {frame * save_every_w * dt_w:.1f} ms"
    )
    plt.show()


interact(
    run_bistable,
    alpha_w=FloatSlider(min=0.05, max=0.50, step=0.01, value=0.10, description="α"),
    k_w=FloatSlider(min=0.5, max=5.0, step=0.1, value=2.0, description="k"),
    A_w=FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description="A"),
    frame=IntSlider(min=0, max=140, step=1, value=70, description="Time frame"),
);